# NGN/USD Exchange Rate Analysis — 2019 to 2024
**Author:** Ojo Timothy  
**Source:** CBN Official Rates (NFEM/NAFEX window) — monthly averages  
**Period:** January 2019 – December 2024

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
    'figure.dpi': 120,
})
NAVY='#1F4E79'; BLUE='#2E75B6'; ORANGE='#E8700A'; RED='#C0392B'; GREEN='#1A7A4A'; GRAY='#95A5A6'
print('Ready.')

## 2. Load & Prepare Dataset

In [ ]:
df = pd.read_csv('data/ngn_usd_exchange_rate_2019_2024.csv', parse_dates=['date'])
df['year'] = df['date'].dt.year
print(f'Dataset loaded: {len(df)} monthly records')
df.head(10)

## 3. Key Summary Statistics

In [ ]:
print('Jan 2019:', f"\u20a6{df['rate_ngn_usd'].iloc[0]:,.1f}")
print('Dec 2024:', f"\u20a6{df['rate_ngn_usd'].iloc[-1]:,.1f}")
print('Total depreciation:', f"{df['cumul_depr_pct'].iloc[-1]:.1f}%")
big = df.loc[df['mom_pct'].idxmax()]
print('Biggest monthly jump:', big['date'].strftime('%B %Y'), f"+{big['mom_pct']:.1f}%")

annual = df.groupby('year')['rate_ngn_usd'].agg(['mean','min','max'])
annual.columns = ['Average','Min','Max']
annual

## 4. Chart 1 — Full Rate Timeline with Rolling Averages

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(df['date'], df['rate_ngn_usd'], color=NAVY, linewidth=2.2, label='Monthly Rate', zorder=3)
ax.fill_between(df['date'], df['rate_ngn_usd'], alpha=0.08, color=BLUE)
ax.plot(df['date'], df['rolling_3m'], color=ORANGE, linewidth=1.4, linestyle='--', label='3-Month Rolling Avg')
ax.plot(df['date'], df['rolling_6m'], color=GREEN,  linewidth=1.4, linestyle=':',  label='6-Month Rolling Avg')

events = [
    ('2020-03', 360, 'COVID-19\nFirst devaluation'),
    ('2021-03', 407, 'CBN\ndevaluation 2'),
    ('2023-06', 770, 'Tinubu FX\nunification'),
    ('2024-02', 1470, 'NFEM rate\nshock'),
]
for m, y, label in events:
    xpos = pd.to_datetime(m)
    ax.axvline(xpos, color=RED, linewidth=1, linestyle='--', alpha=0.5)
    ax.annotate(label, xy=(xpos, y), xytext=(xpos, y+100),
                fontsize=7.5, color=RED, ha='center',
                arrowprops=dict(arrowstyle='-', color=RED, alpha=0.5))

ax.axvspan(pd.to_datetime('2023-06'), df['date'].max(), alpha=0.06, color=ORANGE)
ax.text(pd.to_datetime('2023-09'), 150, 'Float Era', fontsize=8, color=ORANGE)
ax.set_title('NGN/USD Exchange Rate — January 2019 to December 2024', fontsize=14, fontweight='bold', color=NAVY)
ax.set_ylabel('Naira per 1 US Dollar (\u20a6)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'\u20a6{x:,.0f}'))
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 5. Chart 2 — Month-on-Month % Change

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
colors = [RED if v > 0 else GREEN for v in df['mom_pct'].fillna(0)]
ax.bar(df['date'], df['mom_pct'].fillna(0), color=colors, width=20, edgecolor='white', alpha=0.85)
ax.axhline(0, color=GRAY, linewidth=0.8)
top3 = df.nlargest(3,'mom_pct')
for _, row in top3.iterrows():
    ax.text(row['date'], row['mom_pct']+0.5, f"+{row['mom_pct']:.1f}%", ha='center', fontsize=7.5, color=RED, fontweight='bold')
ax.set_title('Month-on-Month % Change in NGN/USD Rate (2019–2024)', fontsize=13, fontweight='bold', color=NAVY)
ax.set_ylabel('% Change')
plt.tight_layout(); plt.show()

## 6. Chart 3 — Year-on-Year Depreciation

In [ ]:
yoy_df = df.dropna(subset=['yoy_pct'])
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(yoy_df['date'], yoy_df['yoy_pct'], color=[RED if v>0 else GREEN for v in yoy_df['yoy_pct']], width=20, edgecolor='white', alpha=0.85)
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Year-on-Year Naira Depreciation vs USD (2020–2024)', fontsize=13, fontweight='bold', color=NAVY)
ax.set_ylabel('YoY % Change')
plt.tight_layout(); plt.show()

## 7. Chart 4 — Annual Average with Min-Max Range

In [ ]:
annual2 = df.groupby('year')['rate_ngn_usd'].agg(['mean','min','max']).reset_index()
annual2.columns = ['year','avg','low','high']
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(annual2))
ax.bar(x, annual2['avg'], color=BLUE, edgecolor='white', width=0.5)
ax.errorbar(x, annual2['avg'], yerr=[annual2['avg']-annual2['low'], annual2['high']-annual2['avg']],
            fmt='none', color=NAVY, capsize=6, linewidth=1.5)
ax.set_xticks(list(x)); ax.set_xticklabels(annual2['year'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'\u20a6{x:,.0f}'))
ax.set_title('Annual Average NGN/USD with Min–Max Range (2019–2024)', fontsize=12, fontweight='bold', color=NAVY)
for i, row in annual2.iterrows():
    ax.text(i, row['avg']+20, f'\u20a6{row["avg"]:,.0f}', ha='center', fontsize=8.5, color=NAVY)
plt.tight_layout(); plt.show()

## 8. Chart 5 — Cumulative Depreciation from 2019 Baseline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df['date'], df['cumul_depr_pct'], color=RED, linewidth=2.2)
ax.fill_between(df['date'], df['cumul_depr_pct'], alpha=0.1, color=RED)
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Cumulative Naira Depreciation from January 2019 Baseline (\u20a6306.9)', fontsize=12, fontweight='bold', color=NAVY)
ax.set_ylabel('Cumulative % Depreciation')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
final = df['cumul_depr_pct'].iloc[-1]
ax.text(df['date'].iloc[-1], final+5, f'Total: +{final:.0f}%', ha='right', fontsize=9, color=RED, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Conclusions

- The Naira depreciated **+400%** against the USD between January 2019 (₦306.9) and December 2024 (₦1,535).
- The single biggest shock was **June 2023 (+66.7%)** — triggered by the Tinubu administration's removal of the multi-tier exchange rate system.
- The 2024 annual average of **₦1,495/USD** was nearly **5x the 2019 average** of ₦306.9.
- Three distinct phases are visible: a managed peg (2019–May 2023), a unification shock (June–Dec 2023), and a volatile float era (2024).
- Rolling averages confirm that depreciation is structural rather than temporary — there has been no sustained recovery toward pre-2023 levels.